# Comprobación integridad referencial archivos movieLens



- Verificar que la columna movieId de tags.parquet y ratings.parquet existe en movies.parquet
- verificar que la columna userId de tags.parquet está en userId de ratings.parquet

In [18]:
import pandas as pd

Se cargan todos los datasets

In [19]:
movies = pd.read_parquet("../data/02_processed/movies_clean.parquet")
tags = pd.read_parquet("../data/02_processed/tags_clean.parquet")
ratings = pd.read_parquet("../data/02_processed/ratings_clean.parquet")
links = pd.read_parquet("../data/02_processed/links_clean.parquet")

Se seleccionan las columnas userId y movieId de los respectivos datasets y se comprueba la identidad referencial

In [20]:
movieId_movies = movies['movieId']
movieId_tags = tags['movieId']
movieId_ratings = ratings['movieId']
movieId_links = links['movieId']

userId_tags = tags['userId']
userId_ratings = ratings['userId']

In [21]:
ratings[~movieId_ratings.isin(movieId_movies)]
# cumple 

,userId,movieId,rating,timestamp


In [22]:
links[~movieId_links.isin(movieId_movies)]
# cumple 
# se conoce de pasos anteriores la existencia de registros en movies que no están en links

,movieId,tmdbId


In [23]:
tags[~movieId_tags.isin(movieId_movies)]
# cumple

,userId,movieId,tag,timestamp


In [24]:
tags[~movieId_tags.isin(movieId_ratings)]
# hay películas que tienen calificación en tags pero no tienen ratings asociado. No influye

,userId,movieId,tag,timestamp
560,288,7020,notable nudity,2006-06-22 14:54:37
584,318,30892,animation,2009-04-19 16:29:50
585,318,30892,documentary,2009-04-19 16:29:11
586,318,30892,henry darger,2009-04-19 16:29:20
1275,474,1076,governess,2006-01-17 18:09:54
1705,474,2939,in netflix queue,2006-01-14 01:29:00
1778,474,3338,in netflix queue,2006-01-14 01:21:20
1798,474,3456,in netflix queue,2006-01-14 01:17:43
1887,474,4194,in netflix queue,2006-01-14 01:19:35
2031,474,5721,in netflix queue,2006-01-14 01:09:36


In [25]:
ratings[~movieId_ratings.isin(movieId_tags)]
# Existen películas que tienen calificaciones en ratings que no tienen calificaciones en tags. No influye

,userId,movieId,rating,timestamp
2,1,6,4.0,2000-07-30 18:37:04
5,1,70,3.0,2000-07-30 18:40:00
8,1,151,5.0,2000-07-30 19:07:21
9,1,157,5.0,2000-07-30 19:08:20
10,1,163,5.0,2000-07-30 19:00:50
...,...,...,...,...
100828,610,163981,3.5,2017-05-03 22:22:35
100830,610,166528,4.0,2017-05-04 06:29:25
100831,610,166534,4.0,2017-05-03 21:53:22
100833,610,168250,5.0,2017-05-08 19:50:47


No se considera que sea un problema de identidad referencial, ya que presentan eventos diferentes y se usarán para distintos tipos de recomendaciones.

In [26]:
tags[~userId_tags.isin(userId_ratings)]
# todos los usuarios que tienen calificaiones en tags tambien tienen calificaciones en ratings

,userId,movieId,tag,timestamp


In [27]:
ratings[~userId_ratings.isin(userId_tags)]
# existen usuarios que solo tienen calificaciones en ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,2000-07-30 18:45:03
1,1,3,4.0,2000-07-30 18:20:47
2,1,6,4.0,2000-07-30 18:37:04
3,1,47,5.0,2000-07-30 19:03:35
4,1,50,5.0,2000-07-30 18:48:51
...,...,...,...,...
99529,609,892,3.0,1996-11-05 19:11:20
99530,609,1056,3.0,1996-11-05 19:11:20
99531,609,1059,3.0,1996-11-05 19:10:54
99532,609,1150,4.0,1996-11-05 19:10:54


Conclusión: todas los identificadores de películas de los ficheros ratings y tags deben existir en movies_clean.parquet, en caso de que no se cumpla eliminar calificaciones asociadas a películas no existentes.   

El archivo requestTMDB.log contiene los ids que habían fallado al realizar la petición.  Estos archivos no cuentan con datos de  películas en formato JSON, para que no haya problemas de integridad se eliminarán las filas asociadas en los todos archivos.

movieId de películas que fallaron 

In [28]:
logs = pd.read_csv("../logs/requestTMDB.log", header = None, sep="|", names = ['timestamp','levelname','tmdbId', 'message'])
logs.head()

,timestamp,levelname,tmdbId,message
0,"2026-06-06 18:42:39,035",WARNING,tmdbId = 876,HTTP_Status = 404
1,"2026-06-06 18:44:47,553",WARNING,tmdbId = 2670,HTTP_Status = 404
2,"2026-06-06 18:55:27,134",WARNING,tmdbId = 7096,HTTP_Status = 404
3,"2026-06-06 18:56:18,174",WARNING,tmdbId = 8677,HTTP_Status = 404
4,"2026-06-06 18:58:14,545",WARNING,tmdbId = 9795,HTTP_Status = 404


In [ ]:
# se extraen  lod id de la columna tmdbId
tmdbId_logs = logs['tmdbId'].str.extract("([0-9]+)").astype('Int64')

#se sacan los movieId asociados a los tmdbId
movieIds_logs = links.loc[links['tmdbId'].isin(tmdbId_logs[0]),'movieId'].tolist()
movieIds_logs


[4207,
 4568,
 5069,
 5209,
 7646,
 7669,
 7762,
 7841,
 7842,
 26453,
 26614,
 26649,
 26693,
 26761,
 26849,
 26887,
 27002,
 27036,
 27251,
 27611,
 27708,
 27751,
 31193,
 38198,
 49917,
 51024,
 52281,
 53883,
 55207,
 57772,
 61406,
 62336,
 62970,
 63433,
 64167,
 65135,
 65359,
 66544,
 66934,
 69524,
 69849,
 70521,
 71160,
 72982,
 77177,
 84847,
 85780,
 86237,
 86487,
 90647,
 90863,
 90945,
 92475,
 93006,
 93008,
 93040,
 93988,
 95717,
 95738,
 96471,
 96518,
 96520,
 99532,
 99764,
 100044,
 100553,
 105250,
 106642,
 107780,
 108727,
 115969,
 119218,
 121035,
 122260,
 122926,
 126430,
 127180,
 127390,
 130842,
 131724,
 137859,
 139130,
 140481,
 140737,
 141816,
 142115,
 147250,
 147328,
 147330,
 148675,
 150548,
 151763,
 152284,
 159817,
 163809,
 167570,
 170355,
 170705,
 171011,
 171495,
 171749,
 172497,
 172909,
 173535,
 173873,
 174053,
 174403,
 175693,
 176329,
 179135,
 180263,
 184257,
 185135]

In [31]:
# se eliminan de todos los archivos los registros asociados a estas peliculas
links = links[~links['movieId'].isin(tmdbId_logs[0])]
ratings = ratings[~ratings['movieId'].isin(tmdbId_logs[0])]
tags = tags[~tags['movieId'].isin(tmdbId_logs[0])]
movies = movies[~movies['movieId'].isin(tmdbId_logs[0])]

Se guardan nuevas versiones de todos los datasets

In [33]:
links.to_parquet('../data/02_processed/links_integrity.parquet', index=False)
ratings.to_parquet('../data/02_processed/ratings_integrity.parquet', index=False)
tags.to_parquet('../data/02_processed/tags_integrity.parquet', index=False)
movies.to_parquet('../data/02_processed/movies_integrity.parquet', index=False)